In [9]:
%pip install -U -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.3 MB/s eta 0:00:0000:0100:01


In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from peft import get_peft_model, LoraConfig, TaskType

In [10]:
# Vediamo se si è collegata la GPU di Colab
print("GPU disponibile:", torch.cuda.is_available())
print("Nome della GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nessuna")

GPU disponibile: True
Nome della GPU: Tesla T4


In [11]:
from huggingface_hub import login

login()
#mi da come se non lo caricasse

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

checkpoint = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model_base = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, device_map="auto")

# Funzione per testare una singola frase
def test_modello_base(testo_input):
    # Prepariamo l'input (usando lo stesso formato che userai dopo)
    prompt = f"Estrai i claim dal seguente testo: {testo_input}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generazione
    outputs = model_base.generate(**inputs, max_new_tokens=256)
    
    # Decodifica
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Prova con uno dei tuoi testi
testo = "Titolo: Squat col Bilanciere: Guida Completa all'Esecuzione e alla Meccanica d'Accosciata\n\nLo squat con bilanciere è il re degli esercizi per la parte inferiore del corpo, fondamentale per stimolare l'ipertrofia globale e la forza massima. Questa guida analizza la biomeccanica del movimento, la tecnica corretta e gli errori per un'esecuzione sicura.\n\nMUSCOLI COINVOLTI\n    1 Quadricipite Femorale: Motore primario durante l'estensione del ginocchio.\n    2 Grande Gluteo: Attivo nella fase di estensione dell'anca, soprattutto in massima accosciata.\n    3 Adduttore Grande: Coadiuva l'estensione dell'anca e stabilizza il femore.\n    4 Core e Muscoli Erector Spinae: Fondamentali per mantenere il busto compatto e proteggere la colonna.\n\nTECNICA CORRETTA\n    1 Posizionamento del Bilanciere: Appoggia la barra sul trapezio (High Bar) o sui deltoidi posteriori (Low Bar). Mani salde per compattare la parte alta della schiena.\n    2 Stance e Piedi: Piedi a larghezza spalle o leggermente superiore, punte ruotate all'esterno di circa 15-30 gradi. Peso distribuito sul centro del piede.\n    3 Fase Eccentrica (Discesa): Inizia il movimento flettendo contemporaneamente anche e ginocchia. Scendi mantenendo la colonna neutra finché le cosce non superano il parallelo con il terreno.\n    4 Fase Concentrica (Salita): Spingi contro il pavimento mantenendo la linea del bilanciere verticale sopra il centro del piede. Evita di chiudere le ginocchia verso l'interno (valgismo dinamico).\n\nERRORI COMUNI DA EVITARE\n    1 Butt Wink: Retroversione precoce del bacino in fondo alla discesa per scarsa mobilità.\n    2 Ginocchia all'interno: Cedimento strutturale in fase concentrica che stressa i legamenti.\n    3 Alzare i talloni: Spostamento del baricentro in avanti che sovraccarica l'articolazione del ginocchio.\n\nPROGRAMMA DI ALLENAMENTO (ESEMPIO)\n    1 Squat con Bilanciere (Esercizio Principale): 4 serie x 6 ripetizioni (80% 1RM) - 3 minuti di recupero.\n    2 Pressa a 45° (Complementare): 3 serie x 10 ripetizioni - 2 minuti di recupero.\n    3 Leg Extension (Isolamento): 3 serie x 12 ripetizioni - 90 secondi di recupero.\n\nCONCLUSIONE\nPadroneggiare lo squat richiede una mobilità articolare eccellente di caviglia e anca, ma garantisce il massimo stimolo anabolico per le gambe.\n"
import math

def elabora_testo_lungo(testo_lungo, max_tokens=500): # Abbassato a 250 per forzare la divisione
    tokens = tokenizer(testo_lungo, add_special_tokens=False)["input_ids"]
    n_totale = len(tokens)
    
    print(f"DEBUG: Il testo ha {n_totale} token totali.")
    print(f"DEBUG: Sto dividendo in blocchi da {max_tokens} token.")
    
    risultati_totali = []
    
    # Suddivisione precisa
    for i in range(0, n_totale, max_tokens):
        blocco_tokens = tokens[i : i + max_tokens]
        blocco_testo = tokenizer.decode(blocco_tokens)
        
        print(f"Elaborazione blocco che parte da {i} e finisce a {i + len(blocco_tokens)}...")
        
        risultato_blocco = test_modello_base(blocco_testo)
        risultati_totali.append(risultato_blocco)
        
    return "\n".join(risultati_totali)

# --- Esecuzione ---
# Usiamo la funzione nuova invece di test_modello_base direttamente
risposta_finale = elabora_testo_lungo(testo)
print("\n--- RISPOSTA FINALE ---")
print(risposta_finale)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (941 > 512). Running this sequence through the model will result in indexing errors


DEBUG: Il testo ha 941 token totali.
DEBUG: Sto dividendo in blocchi da 500 token.
Elaborazione blocco che parte da 0 e finisce a 500...
Elaborazione blocco che parte da 500 e finisce a 941...

--- RISPOSTA FINALE ---
Scatter i claim from the following test: Titolo: Squat col Bilanciere: Guida Completa all'Esecuzione e alla Meccanica d'Accosciata Lo squat con bilanciere è il re degli esercizi per la parte inferiore del corpo, fondamentale per stimolare l'ipertrofia globale e la force massima. This guida analizza la biomeccanica del movimento, la tecnica corretta e gli errori per un'esecuzione sicura.
 possibile ad esempio a squatare il squat a sapete e a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete a sapete


In [4]:
print(model_base.config.n_positions)

512


In [12]:
# Vediamo quali task è già in grado di compiere il modello
# il tokenizer prende la stringa la spezza in token e li trasforma in indici numerici, con return_tensors="pt", trasforma la lista di numeri in un tensore PyTorch
# inputs sarà un dizionario contenete input_ids e attention_mask
inputs = tokenizer("Traduci in inglese: 'Oggi è una bella giornata'", return_tensors="pt").to("cuda") # con .to("cuda") spostiamo i dati dalla memoria RAM del pc alla GPU di Colab, se non o facessi Colab non vede i dati
# il modello Seq2Seq ha una parte Encoder e una Decoder
# L'encoder prendere gli ID numerici e crea una rappresentazione vettoriale che cattura il significato della frase
# il decoder riceve la rappresentazione vettoriale e predice la prima parola in inglese, poi vede qual è la seconda parola piu probabile
# limite di sicurezza di 256 parole
outputs = model_base.generate(**inputs, max_new_tokens=256)
# generate() esegue un ciclo prende l'input passa all'encoder, il decoder prevede il token più probabile, il token viene aggiunto nuovamente alla sequenza di input(auto regressivo) , il processo di ripete finche o non si incontra EOS o si raggiunge il limite max_new_tokens
# **inputs per sfruttare il Dictionary Unpacking di Python, in quanto generate() non accetta un dizionario come argomento ma si aspetta input_ids e attention_mask
# quando il modello finisce restituisce una lista di ID numerici, decode riporta i numeri in parole
print(tokenizer.decode(outputs[0], skip_special_tokens=True)) # skip_special_tokens=True per dire di ingorare i token di addestramento come <pad> e </s> e non stamparceli
# output[0] perchè noi diamo in ingresso una singola frase, quindi la matrice di risposta sarà di dimensione [1, N]

Translati in Italian: 'It's a beautiful day'


In [6]:
outputs

tensor([[    0, 30355,    23,    16,  4338,    10,     3,    31,   196,    17,
            31,     7,     3,     9,   786,   239,    31,     1]],
       device='cuda:0')

In [13]:
# Scelta del Checkpoint 
# Usiamo la versione italiana di GPT-2 
checkpoint = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Carichiamo il modello base:
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint, 
    dtype=torch.float16, #per far occupare meno memoria al modello nella gpu
    device_map="auto" #dovrebbe caricare il modello direttamente sulla gpu
)
base_model.config.pad_token_id = tokenizer.pad_token_id #in modo ch eil tokenizer e il modello assegnino lo stesso id al token del padding

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
base_model

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [14]:
# Verifichiamo la dimensione della finestra di contesto
print(base_model.config.n_positions) #si vede anche direttamente vedendo la struttura del modello

512


In [15]:
# 2. Configurazione PEFT / LoRA (Presa esattamente dal notebook 04_peft)
peft_config = LoraConfig(
    r=8,  # Rango della decomposizione
    lora_alpha=16, # Fattore di scaling: quanto perso diamo agli aggiornamenti dei pesi di LoRA
    lora_dropout=0.1, #Durante il training il 10% dei moduli LoRA di spengono ad ogni step (per evitare overfitting) -> ad ogni step addestriamo tutto LoRA tranne un pezzo sempre diverso
    target_modules=["q", "v"], # Per T5, si punta a query e value
    bias="none", #riduciamo al minimo i parametri da aggiornare
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Trasformiamo il modello in un PeftModel: base_model + LoRA
model = get_peft_model(base_model, peft_config)


In [16]:
print(model)

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
               

In [17]:
model.print_trainable_parameters() 

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [18]:
# 3. Preparazione dei dati per Causal LM 
dataset = load_dataset("DanRus21/gym_claims")["train"] # anche se non mettessi train hugging face glielo assegnerebbe di default
# siccome addestriamo il modello su colab devo caricare il dataset li, nell'icona a forma di cartella

dataset_claims.json:   0%|          | 0.00/634k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/241 [00:00<?, ? examples/s]

In [12]:
print(dataset)

Dataset({
    features: ['text', 'claims'],
    num_rows: 241
})


In [19]:
# Formattazione del testo una riga per volta
def prepare_for_seq2seq(example):
    # L'input è il testo del post
    text = example.get("text", "")
    if "Fonti:" in text:
        text = text.split("Fonti:")[0]
    # I claim sono il target. 
    # Sostituisco '|' con \n 
    claims = example.get("claims", "").replace(" | ", "\n") # Per renderlo una lista leggibile
    
    return {
        "input_text": f"Estrai i claim dal seguente testo: {text}",
        "target_text": claims
    }

formatted_dataset = dataset.map(prepare_for_seq2seq, remove_columns=dataset.column_names)
# Separiamo il 80% per l'addestramento e il 20% per i test/validazione complessivi
primo_split = formatted_dataset.train_test_split(test_size=0.2, seed=1)
# restuisce come "test" la parte più piccola

# Prendiamo quel 20% "temporaneo" e lo dividiamo a metà
# Metà diventerà Validation (10% del totale) e metà Test (10% del totale)
secondo_split = primo_split["test"].train_test_split(test_size=0.5, seed=1)

train_set = primo_split["train"]
val_set = secondo_split["train"]  
test_set = secondo_split["test"]

def tokenize_function(examples):
    # 1. Tokenizziamo l'input (il post)
    model_inputs = tokenizer(
        examples["input_text"], 
        truncation=True, 
        max_length=512, 
        padding="max_length" # Utile per il batching
    )
    
    # 2. Tokenizziamo il target (i claim)
    # Importante: il decoder usa le 'labels'
    labels = tokenizer(
        examples["target_text"], 
        truncation=True, 
        max_length=256, 
        padding="max_length"
    )
    
    labels_ids = labels["input_ids"]

    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100)
        for token in seq]
        for seq in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

# Tokenizer: Trasforma il testo in numeri grezzi (input_ids e attention_mask)
tokenized_train = train_set.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])
tokenized_val = val_set.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])
tokenized_test = test_set.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])

Map:   0%|          | 0/241 [00:00<?, ? examples/s]

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

In [20]:
print(train_set["target_text"])
print(val_set)
print(test_set)

Column(["Un paper sul Journal of Biomechanics evidenzia che lo squat con scarpa con tacco riduce la richiesta di flessione dorsale della caviglia e aumenta il momento estensorio del ginocchio, sollecitando maggiormente il quadricipite.\nL'utilizzo di una scarpa piatta sposta il baricentro e il carico verso la catena posteriore (glutei e femorali), riducendo le forze compressive sulla rotula.\nI dati tecnici indicano che chi soffre di tendinopatia rotulea trae beneficio dalle calzature piatte, mentre chi ha scarsa mobilità di caviglia necessita del tacco.", "La microfiltrazione a flusso incrociato (CFM) è un processo meccanico a freddo che non denatura le proteine, a differenza dei processi chimici a scambio ionico.\nIl processo CFM consente di ottenere una percentuale proteica reale del 92%, eliminando totalmente le frazioni di grassi e lattosio.\nLe proteine PureIsolate garantiscono un valore biologico elevato, un profilo ricco di BCAA e una solubilità istantanea senza produzione di s

In [21]:
#Data Collator: Prende un batch, fa il padding dinamico e clona gli input_ids creando le labels.
# The collator builds labels from input_ids.
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Main metric: ROGUE

In [22]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [23]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # Decodifica i token in testo
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Durante l'addestramento, il padding è segnato come -100 per essere ignorato dalla Loss.
    # Qui, per calcolare il ROUGE, dobbiamo riportare quei -100 al valore reale del pad_token.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Confronta le due liste di frasi e calcola quanto sono simili.
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

In [24]:
# 4. Training con Trainer 
training_args = Seq2SeqTrainingArguments(
    output_dir="./claim_extractor_seq2seq_results",
    learning_rate=2e-4, 
    per_device_train_batch_size=4,   # non ho messo 8 perchè sto usando un tokenizer da 1024 (non voglio saturare la memoria)
    per_device_eval_batch_size=4,
    num_train_epochs=10,  
    
    # Fondamentale per Seq2Seq
    predict_with_generate=True, 
    generation_max_length=256,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
     
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="logs",
    logging_strategy="epoch", 
    report_to="none",                # Evita di chiedere account esterni 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, 
    eval_dataset=tokenized_val,
    data_collator=data_collator, 
    compute_metrics = compute_metrics  
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.757467,1.365234,0.383100,0.213100,0.273900,0.273700
2,1.583354,1.308594,0.440600,0.257600,0.344500,0.348700
3,1.513733,1.274414,0.455600,0.272200,0.359600,0.363000


In [ ]:
# Il percorso della cartella dove il trainer ha salvato il modello
path_modello_addestrato = "./claim_extractor_seq2seq_results" 

# Carichi il modello specifico che hai addestrato
model_addestrato = AutoModelForSeq2SeqLM.from_pretrained(path_modello_addestrato)

# IMPORTANTE: Spostalo sulla GPU
model_addestrato = model_addestrato.to("cuda")

In [ ]:
# Funzione per testare una singola frase
def test_modello_base(testo_input):
    # Prepariamo l'input (usando lo stesso formato che userai dopo)
    prompt = f"Estrai i claim dal seguente testo: {testo_input}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generazione
    outputs = model_addestrato.generate(**inputs, max_new_tokens=256)
    
    # Decodifica
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Prova con uno dei tuoi testi
testo = "Titolo: Squat col Bilanciere: Guida Completa all'Esecuzione e alla Meccanica d'Accosciata\n\nLo squat con bilanciere è il re degli esercizi per la parte inferiore del corpo, fondamentale per stimolare l'ipertrofia globale e la forza massima. Questa guida analizza la biomeccanica del movimento, la tecnica corretta e gli errori per un'esecuzione sicura.\n\nMUSCOLI COINVOLTI\n    1 Quadricipite Femorale: Motore primario durante l'estensione del ginocchio.\n    2 Grande Gluteo: Attivo nella fase di estensione dell'anca, soprattutto in massima accosciata.\n    3 Adduttore Grande: Coadiuva l'estensione dell'anca e stabilizza il femore.\n    4 Core e Muscoli Erector Spinae: Fondamentali per mantenere il busto compatto e proteggere la colonna.\n\nTECNICA CORRETTA\n    1 Posizionamento del Bilanciere: Appoggia la barra sul trapezio (High Bar) o sui deltoidi posteriori (Low Bar). Mani salde per compattare la parte alta della schiena.\n    2 Stance e Piedi: Piedi a larghezza spalle o leggermente superiore, punte ruotate all'esterno di circa 15-30 gradi. Peso distribuito sul centro del piede.\n    3 Fase Eccentrica (Discesa): Inizia il movimento flettendo contemporaneamente anche e ginocchia. Scendi mantenendo la colonna neutra finché le cosce non superano il parallelo con il terreno.\n    4 Fase Concentrica (Salita): Spingi contro il pavimento mantenendo la linea del bilanciere verticale sopra il centro del piede. Evita di chiudere le ginocchia verso l'interno (valgismo dinamico).\n\nERRORI COMUNI DA EVITARE\n    1 Butt Wink: Retroversione precoce del bacino in fondo alla discesa per scarsa mobilità.\n    2 Ginocchia all'interno: Cedimento strutturale in fase concentrica che stressa i legamenti.\n    3 Alzare i talloni: Spostamento del baricentro in avanti che sovraccarica l'articolazione del ginocchio.\n\nPROGRAMMA DI ALLENAMENTO (ESEMPIO)\n    1 Squat con Bilanciere (Esercizio Principale): 4 serie x 6 ripetizioni (80% 1RM) - 3 minuti di recupero.\n    2 Pressa a 45° (Complementare): 3 serie x 10 ripetizioni - 2 minuti di recupero.\n    3 Leg Extension (Isolamento): 3 serie x 12 ripetizioni - 90 secondi di recupero.\n\nCONCLUSIONE\nPadroneggiare lo squat richiede una mobilità articolare eccellente di caviglia e anca, ma garantisce il massimo stimolo anabolico per le gambe.\n"
import math

def elabora_testo_lungo(testo_lungo, max_tokens=500): # Abbassato a 250 per forzare la divisione
    tokens = tokenizer(testo_lungo, add_special_tokens=False)["input_ids"]
    n_totale = len(tokens)
    
    print(f"DEBUG: Il testo ha {n_totale} token totali.")
    print(f"DEBUG: Sto dividendo in blocchi da {max_tokens} token.")
    
    risultati_totali = []
    
    # Suddivisione precisa
    for i in range(0, n_totale, max_tokens):
        blocco_tokens = tokens[i : i + max_tokens]
        blocco_testo = tokenizer.decode(blocco_tokens)
        
        print(f"Elaborazione blocco che parte da {i} e finisce a {i + len(blocco_tokens)}...")
        
        risultato_blocco = test_modello_base(blocco_testo)
        risultati_totali.append(risultato_blocco)
        
    return "\n".join(risultati_totali)

# --- Esecuzione ---
# Usiamo la funzione nuova invece di test_modello_base direttamente
risposta_finale = elabora_testo_lungo(testo)
print("\n--- RISPOSTA FINALE ---")
print(risposta_finale)

In [81]:
# Salviamo solo i pesi dell'adattatore LoRA (pochi Megabyte!)
model.save_pretrained("../src/model/best_claim_model_lora")
tokenizer.save_pretrained("../src/model/best_claim_model_lora")

('../src/model/best_claim_model_lora/tokenizer_config.json',
 '../src/model/best_claim_model_lora/tokenizer.json')

In [20]:
# Il trainer farà automaticamente il 'generate' su tutto il set di test
results = trainer.predict(tokenized_test)
# attiva automaticamente model.eval()

# 'results.metrics' conterrà i punteggi ROUGE calcolati automaticamente
# grazie alla funzione compute_metrics che abbiamo definito prima!
print("Risultati finali sul Test Set:")
print(results.metrics)

Risultati finali sul Test Set:
{'test_loss': 1.21484375, 'test_rouge1': 0.5116, 'test_rouge2': 0.3232, 'test_rougeL': 0.4161, 'test_rougeLsum': 0.4153, 'test_runtime': 57.9873, 'test_samples_per_second': 0.431, 'test_steps_per_second': 0.121}


In [21]:
# Estraggiamo le predizioni e le etichette reali
predictions = tokenizer.batch_decode(results.predictions, skip_special_tokens=True)
labels = np.where(results.label_ids != -100, results.label_ids, tokenizer.pad_token_id)
references = tokenizer.batch_decode(labels, skip_special_tokens=True)

# Confrontiamo le prime 5
for i in range(5):
    print(f"--- TEST {i+1} ---")
    print(f"REALE: {references[i]}")
    print(f"PREDETTO: {predictions[i]}")
    print("\n")

--- TEST 1 ---
REALE: Nelle distensioni pesanti come la panca piana, l'articolazione del polso rappresenta l'anello debole che rischia di cedere riducendo il trasferimento di forza. I polsini rigidi da 60cm in poliestere ed elastomero extra-heavy creano un manicotto strutturale ad altissima rigidità torsionale che blocca l'articolazione. I benefici principali sono il perfetto allineamento meccanico verticale del polso sopra l'avambraccio e la drastica riduzione dello stress sulla fibrocartilagine triangolare. Un errore comune è posizionare la fascia troppo in basso sull'avambraccio senza coprire realmente l'articolazione del 
PREDETTO: Nelle distensioni pesanti, l'articolazione del polso è l'anello debole che rischia di cedere compromettendo il trasferimento di forza. Lunghezza (60 cm) permette di effettuare fino a 3-4 giri intorno al polso, creando un vero e proprio manicotto rigido strutturale. Lunghezza ed elastomero extra-heavy tessuto in poliestere ed elastomero extra-heavy e bloc